# Phase 3: LLM 기반 UMC 차원 분류 및 감성 분석 (Claude Code Workflow)

이 노트북은 `Phase 1`(`notebooks/phase1_keyword_filtering.ipynb`)과 `Phase 0`(`src/sample_for_claude.py`)의 패러다임을 결합하여, 불필요한 데이터 중복 복사 없이 원본(split_by_gu)을 참조하되 최적화된 프롬프트를 생성합니다.

- **테스트 모드 지원**: 테스트나 초기 확인을 위해 각 자치구별로 n개씩만 샘플링하여 처리할 수 있습니다. (전체 실행 시 None 지정)
- **Claude 지시문**: `prompts/phase03/prompt_*.md` (원하는 결과를 얻기 위해 데이터 일부를 예시로 포함한 정밀 마크다운 프롬프트)
- **최종 결과**: `data/processed/phase03_analyzed/*.csv`

In [ ]:
import os
import sys
from pathlib import Path
from tqdm.notebook import tqdm

# 현재 작업 디렉토리를 프로젝트 루트로 설정
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"현재 작업 디렉토리: {Path.cwd()}")

sys.path.append(str(Path.cwd()))

## 1. 프롬프트 가이드 생성 및 데이터 샘플링 (선택)

`SAMPLE_SIZE` 변수를 통해 각 자치구에서 추출할 테스트용 데이터 개수를 지정할 수 있습니다. 
- `SAMPLE_SIZE = 30` 이면 각 자치구별 30개씩만 임시 추출하여 `data/processed/phase03_test_samples/` 에 저장하고, Claude가 이를 읽도록 프롬프트를 만듭니다.
- `SAMPLE_SIZE = None` 이면 파일 복사 없이 대용량의 원본(split_by_gu)을 직접 참고하도록 프롬프트를 만듭니다.

In [ ]:
from src.phase03_llm_analysis import run_prepare

ORIGINAL_GU_DIR = Path("data/processed/split_by_gu")
PROMPT_OUTPUT_DIR = Path("prompts/phase03")

# 테스트 및 확인용: 각 구별로 추출할 데이터 개수를 입력하세요. 전체 데이터에 대해 실행하려면 None으로 둡니다.
SAMPLE_SIZE = 30  

gu_csv_files = list(ORIGINAL_GU_DIR.glob("*.csv"))
print(f"총 {len(gu_csv_files)}개의 자치구 데이터 파일이 발견되었습니다.")

for csv_file in tqdm(gu_csv_files, desc="프롬프트 생성 중"):
    run_prepare(
        input_path=csv_file,
        output_prompt_dir=PROMPT_OUTPUT_DIR,
        sample_size=SAMPLE_SIZE
    )

## 2. (수동) Claude Code CLI 실행

터미널을 열고 생성된 `prompts/phase03/` 폴더 안의 내용들을 통해 Claude Code로 JSON 분석을 요청합니다.

터미널 안내:
```bash
> cd data/processed/split_by_gu
> claude
```

명령 예시 (전체 자동화):
> "`../../../prompts/phase03/`에 있는 각 구별 명령 마크다운 파일들을 순차적으로 참고해서 지시된 CSV 파일들을 분석해줘. 출력 JSON은 반드시 `../../../prompts/phase03_responses/` 안에 구 이름과 동일하게(예: 종로구.json) 저장해줘."

*(Claude CLI 에이전트가 프롬프트 내에 기재된 `target_csv` 경로를 스스로 찾아가 분석합니다)*

## 3. 분석 결과 병합 (Merge)

Claude Code가 모든 파일 처리를 끝마쳤다면 `prompts/phase03_responses/` 폴더에 생성된 구별 JSON(`.json`) 파일들을 원래 분할된 원본 데이터 CSV에 일괄 병합합니다.

In [ ]:
from src.phase03_llm_analysis import run_merge

RESPONSES_DIR = Path("prompts/phase03_responses")
FINAL_OUTPUT_DIR = Path("data/processed/phase03_analyzed")

json_files = list(RESPONSES_DIR.glob("*.json"))
print(f"총 {len(json_files)}개의 JSON 응답 파일을 찾아 병합을 시작합니다.")

if len(json_files) == 0:
    print("⚠️ 응답 폴더에 JSON 파일이 없습니다. Claude Code 실행 후 다시 시도해주세요.")
else:
    for json_file in tqdm(json_files, desc="결과 병합 중"):
        # JSON 응답 파일명(예: 종로구.json)과 원본 데이터 파일명(예: 종로구.csv) 매핑
        gu_name = json_file.stem
        target_original_csv = ORIGINAL_GU_DIR / f"{gu_name}.csv"
        target_output_csv = FINAL_OUTPUT_DIR / f"{gu_name}.csv"
        
        run_merge(
            input_path=target_original_csv,
            response_path=json_file,
            output_path=target_output_csv
        )
    print("\n✅ 모든 자치구 병합 완료!")